# Data Retrieval Benchmark: Flat Table vs Relational (Star Schema)

**Question:** For dynamic dashboard filtering in Python, is it faster to filter a single flat (denormalised) DataFrame, or to filter via dimension keys and retrieve from the fact table?

We test both approaches across multiple filter scenarios and measure execution time. The winning approach is used in the dashboard.

---

## 1. Setup

In [8]:
import sys
import time
import warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
from pathlib import Path

warnings.filterwarnings('ignore')

root_path = Path(os.getcwd()).parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))
from src.data_modelling import load_and_prepare

data_path = root_path / 'data' / 'Company_X_Audience.xlsx'

fact, dim_customer, dim_date, dim_seating, analysis = load_and_prepare(data_path)

print(f'FactTransaction : {fact.shape[0]:,} rows × {fact.shape[1]} cols')
print(f'DimCustomer     : {dim_customer.shape[0]:,} rows × {dim_customer.shape[1]} cols')
print(f'DimDate         : {dim_date.shape[0]:,} rows × {dim_date.shape[1]} cols')
print(f'DimSeating      : {dim_seating.shape[0]:,} rows × {dim_seating.shape[1]} cols')
print(f'Flat (analysis) : {analysis.shape[0]:,} rows × {analysis.shape[1]} cols')

FactTransaction : 800 rows × 12 cols
DimCustomer     : 800 rows × 6 cols
DimDate         : 242 rows × 8 cols
DimSeating      : 4 rows × 4 cols
Flat (analysis) : 800 rows × 27 cols


---
## 2. The Two Approaches

### Approach A — Flat Table Filter
Filter the single denormalised `analysis` DataFrame directly using boolean masks.  
One pass, no joins.

### Approach B — Relational Filter (Star Schema)
Filter each dimension table to get matching surrogate keys, then retrieve matching rows from the fact table using `.isin()` lookups, then re-join dimensions for the result.  
Multiple passes + join overhead.

In [9]:
# ── Approach A: Flat table filter ─────────────────────────────────────────────
def filter_flat(
    analysis: pd.DataFrame,
    countries:  list = None,
    seating:    list = None,
    genders:    list = None,
    age_groups: list = None,
) -> pd.DataFrame:
    """
    Filter the flat denormalised table with boolean masks.
    All filters are AND-combined. None means 'no filter on this dimension'.
    """
    mask = pd.Series(True, index=analysis.index)
    if countries:  mask &= analysis['Country'].isin(countries)
    if seating:    mask &= analysis['Seating_Region'].isin(seating)
    if genders:    mask &= analysis['Gender'].isin(genders)
    if age_groups: mask &= analysis['Age_Group'].isin(age_groups)
    return analysis[mask]


# ── Approach B: Relational filter via dimension keys ──────────────────────────
def filter_relational(
    fact:         pd.DataFrame,
    dim_customer: pd.DataFrame,
    dim_seating:  pd.DataFrame,
    dim_date:     pd.DataFrame,
    countries:    list = None,
    seating:      list = None,
    genders:      list = None,
    age_groups:   list = None,
) -> pd.DataFrame:
    """
    Filter dimension tables to get valid surrogate keys,
    retrieve matching fact rows via .isin(), then rejoin dims.
    """
    # Step 1: filter each dimension to get valid SKs
    cust_dim = dim_customer.copy()
    if countries:  cust_dim = cust_dim[cust_dim['Country'].isin(countries)]
    if genders:    cust_dim = cust_dim[cust_dim['Gender'].isin(genders)]
    if age_groups: cust_dim = cust_dim[cust_dim['Age_Group'].isin(age_groups)]

    seat_dim = dim_seating.copy()
    if seating: seat_dim = seat_dim[seat_dim['Seating_Region'].isin(seating)]

    # Step 2: retrieve matching fact rows using SK lookups
    fact_filtered = fact[
        fact['Customer_SK'].isin(cust_dim['Customer_SK']) &
        fact['Seating_SK'].isin(seat_dim['Seating_SK'])
    ]

    # Step 3: rejoin all dimensions to restore attribute columns
    result = (
        fact_filtered
        .merge(dim_customer, on='Customer_SK')
        .merge(dim_date,     on='Date_SK')
        .merge(dim_seating,  on='Seating_SK')
    )
    return result


print('Both filter functions defined.')

Both filter functions defined.


---
## 3. Quick Correctness Check
Both approaches must return identical row counts for the same filter before we time them.

In [10]:
test_filter = dict(
    countries=['USA', 'Japan'],
    seating=['VIP', 'Premium'],
    genders=['Female'],
    age_groups=None,
)

result_flat = filter_flat(analysis, **test_filter)
result_rel  = filter_relational(
    fact, dim_customer, dim_seating, dim_date,
    **test_filter
)

print(f'Flat result rows       : {len(result_flat)}')
print(f'Relational result rows : {len(result_rel)}')
print(f'Match: {len(result_flat) == len(result_rel)} ✓' if len(result_flat) == len(result_rel) else 'MISMATCH ✗')

Flat result rows       : 54
Relational result rows : 54
Match: True ✓


---
## 4. Benchmark — Multiple Filter Scenarios

We test 6 scenarios that represent realistic dashboard filter interactions:

| Scenario | Description | Selectivity |
|----------|-------------|-------------|
| 1 | No filters (full table) | 100% rows |
| 2 | Single country | ~14% rows |
| 3 | Two countries + one seating | ~14% rows |
| 4 | Gender + age group | ~25% rows |
| 5 | All 4 filter dimensions active | ~5% rows |
| 6 | Single value on every filter | ~2% rows |

In [11]:
RUNS = 500  # number of repetitions per scenario for stable timing

scenarios = [
    {
        'label':      'No filters (full table)',
        'countries':  None, 'seating': None, 'genders': None, 'age_groups': None,
    },
    {
        'label':      'Single country',
        'countries':  ['USA'], 'seating': None, 'genders': None, 'age_groups': None,
    },
    {
        'label':      '2 countries + 1 seating',
        'countries':  ['UK', 'Germany'], 'seating': ['VIP'], 'genders': None, 'age_groups': None,
    },
    {
        'label':      'Gender + age group',
        'countries':  None, 'seating': None, 'genders': ['Female'], 'age_groups': ['25-35', '35-50'],
    },
    {
        'label':      'All 4 dimensions active',
        'countries':  ['USA', 'Japan'], 'seating': ['VIP', 'Premium'],
        'genders':    ['Female'], 'age_groups': ['35-50', '50-65'],
    },
    {
        'label':      'Highly selective (all single)',
        'countries':  ['France'], 'seating': ['Economy'],
        'genders':    ['Male'], 'age_groups': ['Under 25'],
    },
]

results = []

for s in scenarios:
    label       = s['label']
    filter_args = {k: v for k, v in s.items() if k != 'label'}

    # ── Time Approach A ──────────────────────────────────────────────────────
    times_flat = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        r  = filter_flat(analysis, **filter_args)
        times_flat.append(time.perf_counter() - t0)
    
    rows_returned = len(r)

    # ── Time Approach B ──────────────────────────────────────────────────────
    times_rel = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        _  = filter_relational(fact, dim_customer, dim_seating, dim_date, **filter_args)
        times_rel.append(time.perf_counter() - t0)

    flat_mean = np.mean(times_flat)  * 1000   # convert to ms
    rel_mean  = np.mean(times_rel)   * 1000
    flat_std  = np.std(times_flat)   * 1000
    rel_std   = np.std(times_rel)    * 1000
    speedup   = rel_mean / flat_mean

    results.append({
        'Scenario':         label,
        'Rows Returned':    rows_returned,
        'Flat (ms)':        round(flat_mean, 4),
        'Flat Std (ms)':    round(flat_std,  4),
        'Relational (ms)':  round(rel_mean,  4),
        'Relational Std':   round(rel_std,   4),
        'Speedup (×)':      round(speedup,   2),
        'Winner':           'Flat ✓' if flat_mean < rel_mean else 'Relational ✓',
    })
    print(f"[{label}]  Flat: {flat_mean:.4f}ms  |  Relational: {rel_mean:.4f}ms  |  Speedup: {speedup:.2f}×")

df_results = pd.DataFrame(results)
print('\nDone.')

[No filters (full table)]  Flat: 0.1566ms  |  Relational: 3.6190ms  |  Speedup: 23.11×
[Single country]  Flat: 0.8180ms  |  Relational: 4.4699ms  |  Speedup: 5.46×
[2 countries + 1 seating]  Flat: 1.0867ms  |  Relational: 5.0558ms  |  Speedup: 4.65×
[Gender + age group]  Flat: 0.9697ms  |  Relational: 4.5482ms  |  Speedup: 4.69×
[All 4 dimensions active]  Flat: 1.2745ms  |  Relational: 5.5295ms  |  Speedup: 4.34×
[Highly selective (all single)]  Flat: 1.2476ms  |  Relational: 5.1495ms  |  Speedup: 4.13×

Done.


---
## 5. Results Table

In [12]:
display_cols = ['Scenario', 'Rows Returned', 'Flat (ms)', 'Relational (ms)', 'Speedup (×)', 'Winner']
df_results[display_cols].style \
    .format({'Flat (ms)': '{:.4f}', 'Relational (ms)': '{:.4f}', 'Speedup (×)': '{:.2f}×'}) \
    .background_gradient(subset=['Speedup (×)'], cmap='RdYlGn') \
    .set_caption(f'Results averaged over {RUNS} runs per scenario') \
    .set_properties(**{'text-align': 'left'})

,Scenario,Rows Returned,Flat (ms),Relational (ms),Speedup (×),Winner
0,No filters (full table),800,0.1566,3.6190,23.11×,Flat ✓
1,Single country,123,0.8180,4.4699,5.46×,Flat ✓
2,2 countries + 1 seating,24,1.0867,5.0558,4.65×,Flat ✓
3,Gender + age group,193,0.9697,4.5482,4.69×,Flat ✓
4,All 4 dimensions active,27,1.2745,5.5295,4.34×,Flat ✓
5,Highly selective (all single),1,1.2476,5.1495,4.13×,Flat ✓


---
## 6. Visualisation

In [13]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Mean Execution Time per Scenario (ms)',
        'Flat Table Speedup over Relational (×)',
    ),
    column_widths=[0.65, 0.35],
)

short_labels = [
    s['label'].replace(' (full table)', '').replace(' active', '')
    for s in scenarios
]

# ── Left: grouped bar ────────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    name='Flat table',
    x=short_labels, y=df_results['Flat (ms)'],
    error_y=dict(type='data', array=df_results['Flat Std (ms)'].tolist(), visible=True),
    marker_color='#C2185B',
    offsetgroup=0,
), row=1, col=1)

fig.add_trace(go.Bar(
    name='Relational (star schema)',
    x=short_labels, y=df_results['Relational (ms)'],
    error_y=dict(type='data', array=df_results['Relational Std'].tolist(), visible=True),
    marker_color='#1A237E',
    offsetgroup=1,
), row=1, col=1)

# ── Right: speedup bar ───────────────────────────────────────────────────────
speedup_colors = ['#2E7D32' if v >= 1 else '#C62828' for v in df_results['Speedup (×)']]

fig.add_trace(go.Bar(
    name='Speedup (×)',
    x=short_labels, y=df_results['Speedup (×)'],
    marker_color=speedup_colors,
    text=df_results['Speedup (×)'].apply(lambda v: f'{v:.2f}×'),
    textposition='outside',
    showlegend=False,
), row=1, col=2)

fig.add_hline(y=1, line_dash='dash', line_color='#999',
              annotation_text='Break-even', row=1, col=2)

fig.update_layout(
    height=420,
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=0.65),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(family='Inter, Arial, sans-serif', size=12),
    margin=dict(l=10, r=10, t=60, b=80),
)
fig.update_xaxes(tickangle=-25, tickfont_size=10)
fig.update_yaxes(showgrid=True, gridcolor='#f0f0f0', row=1, col=1)
fig.update_yaxes(title_text='Speedup (×)', row=1, col=2)
fig.update_yaxes(title_text='Time (ms)',   row=1, col=1)

fig.show()

---
## 7. Why Flat Wins in Python

The benchmark confirms **Approach A (flat table) is consistently faster** across all filter scenarios. Here is why:

| Factor | Flat Table | Relational (Star Schema) |
|--------|-----------|-------------------------|
| **Passes over data** | 1 (single boolean mask) | 3–4 (filter each dim, isin on fact, merge) |
| **Memory allocation** | Minimal (view/slice) | New DataFrames at each step |
| **Join cost** | None | Paid on every filter change |
| **Pandas internals** | Vectorised C mask | Python-level merge overhead |

**Key insight:** In a proper database (SQL Server, PostgreSQL), the star schema wins because foreign-key indexes allow the engine to skip scanning the full fact table — it jumps directly to matching rows. Pandas has no indexes in that sense. Every `.isin()` call is a full scan anyway, so the relational approach pays join overhead *on top of* the same full scan the flat approach does in one pass.

**Conclusion for the dashboard:** The flat `analysis` DataFrame is used for all dynamic filtering. The star schema tables (`fact`, `dim_customer`, `dim_date`, `dim_seating`) are retained in the codebase to demonstrate correct BI modelling principles — they are the source of truth from which `analysis` is derived — but filtering at runtime always goes through the flat table.

In [14]:
avg_speedup = df_results['Speedup (×)'].mean()
max_speedup = df_results['Speedup (×)'].max()
max_scenario = df_results.loc[df_results['Speedup (×)'].idxmax(), 'Scenario']

print('=== Summary ===')
print(f'Average speedup of flat over relational : {avg_speedup:.2f}×')
print(f'Maximum speedup                         : {max_speedup:.2f}× ({max_scenario})')
print(f'Flat table wins in                      : {(df_results["Winner"] == "Flat ✓").sum()}/{len(df_results)} scenarios')

=== Summary ===
Average speedup of flat over relational : 7.73×
Maximum speedup                         : 23.11× (No filters (full table))
Flat table wins in                      : 6/6 scenarios
